# Phase 3: Existing controlled-evaluation analysis

This notebook consolidates the existing Phase 3 methodology, tables, five revised
charts, comparison interpretation, and historical chart context. It reads saved
results only: no experiment, monitor, test suite, tuning, or new research runs.
The three maintained roles are `phase3_implement.py`, `phase3_test.py`, and this
notebook; legacy entry points were retired at the US-3.32 verified cutover.
US-3.34 keeps the four user-readable outputs separate from three root-level
operational run files: the manifest, complete C0 events, and independent receipt.

**Choose `OUTPUT_DIR` below.** The default is `artifacts/phase3_outputs`; its
paired operational directory is `artifacts/phase3_operational`. Missing, stale,
or mixed-run evidence fails rather than falling back to historical outputs. For
an isolated candidate, set `PHASE3_OUTPUT_DIR` to a directory named
`phase3_outputs` before starting the kernel; its sibling `phase3_operational`
holds run-specific machine evidence. An optional
`PHASE3_REPO_ROOT` selects the corresponding complete checkout. Run all cells from
a clean kernel. Normal execution writes no files and keeps charts inline.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

root_setting = os.environ.get("PHASE3_REPO_ROOT")
if root_setting:
    ROOT = Path(root_setting).resolve()
else:
    candidates = (Path.cwd(), Path.cwd().parent)
    ROOT = next((path for path in candidates if (path / "scripts" / "phase3_implement.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Run from the repository root/scripts directory or set PHASE3_REPO_ROOT.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.phase3_implement import load_phase3_analysis_results

OUTPUT_DIR = Path(os.environ.get("PHASE3_OUTPUT_DIR", str(ROOT / "artifacts" / "phase3_outputs")))
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = ROOT / OUTPUT_DIR
OUTPUT_DIR = OUTPUT_DIR.resolve()
OPERATIONAL_DIR = (OUTPUT_DIR.with_name("phase3_operational") if OUTPUT_DIR.name == "phase3_outputs" else OUTPUT_DIR / "phase3_operational")
print(f"Repository: {ROOT}\nSaved results: {OUTPUT_DIR}\nOperational evidence: {OPERATIONAL_DIR}")

## 1. Locked experiment and validation provenance

The fixed six-sensor A/C/D/E/F/G temperature cohort uses inclusive
2022-02-24 18:50 UTC through 2022-03-10 15:53 UTC: 19,984 consecutive minute
buckets, without imputation. The threshold is an empirical stable-cohort P90,
derived by shared monitoring preparation from the six-sensor mean over this
same evaluation population. It is retrospective, not an external operating or
equipment safety limit. No Phase 1 cohort statistics row or hard-coded threshold
is authoritative; the run evidence records the derived full-precision value.
SAFE means metric < threshold; VIOLATED means metric >= threshold.

Correctness means sampled-bucket root-known-state agreement after that
bucket's protocol events, not continuous-time safety or physical latency.

The read-only loader checks the complete control/alpha grid, schemas,
accounting, current input/code/test hashes, C0 event hashes, matching
completion/failure/run records, and the independently executed validation
receipt **before any scientific table/chart is presented**. It does not rerun
tests or replay monitors. Substantive behavioral/saved/replay verification belongs
to `phase3_test.py`; a saved pass flag alone is not independent proof.

In [ ]:
bundle = load_phase3_analysis_results(ROOT, OUTPUT_DIR)
report = bundle["report"]
controls_frame = bundle["controls"]
alpha_frame = bundle["alphas"]
size_frame = bundle["cluster_sizes"]
control_size_frame = bundle["control_cluster_sizes"]
snapshots = bundle["snapshots"]
equivalence = bundle["equivalence"]
revised = bundle["revised"]
historical = bundle["historical"]
config = bundle["config"]
controls = controls_frame.to_dict(orient="records")
alphas = alpha_frame.to_dict(orient="records")
minima = report["alpha_minima"]
ALPHA_LABELS = tuple(f"{step / 20:.2f}" for step in range(1, 20))
figures = {}
chart_series = {}
input_hashes = bundle["input_hashes"]
display(pd.DataFrame([{
    "run_id": report["run_id"],
    "saved_directory": str(OUTPUT_DIR),
    "sensor_ids": ", ".join(report["experiment"]["sensor_ids"]),
    "buckets": report["experiment"]["bucket_count"],
    "threshold_full_precision": repr(report["experiment"]["threshold"]),
    "validation_checks": ", ".join(report["validation"]["checks"]),
    "executed_test_count": report["validation"]["test_count"],
    "implementation_sha256": report["provenance"]["phase3_implementation_sha256"],
    "test_sha256": json.dumps(report["provenance"]["test_sha256"], sort_keys=True),
}]).T.rename(columns={0: "saved value"}))

## 2. Existing method and source explanations

Clustering uses discounted signed reference-deviation histories, normalized by
the largest absolute history; all-zero histories have a deterministic singleton
fallback. Candidate merges preserve signed cancellation, use the existing
alpha-dependent quality, and select the earliest minimum deterministically.
Histories and raw-reading moments are distinct persistent quantities.
The source-specific FMIC/SMIC interpretation below is retained from the former
implementation notebook; it is explanation, not a second monitoring engine.

### First-moment provenance and recurrence

Source: [Clustering for Monitoring Distributed Data Streams, section 1.4, printed page 10 / PDF page 14](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=14), the discounted vector set immediately before the moment-based node constraints. For observation index `m >= 1`, the source set is `V'_n(m) = {2^(-(m-i)) * v_n(t_i) : i=1,...,m}` and `mu_n` is its arithmetic mean. Section 1.5.2 identifies FMIC as using only the first moment.

Therefore `S_n(m) = sum(i=1..m, 2^(-(m-i)) * v_n(t_i)) = 0.5*S_n(m-1) + v_n(t_m)` and `mu_n(m) = S_n(m)/m`, with `S_n(0)=0`. The recurrence is an algebraic implementation of that source set, not a separately quoted source equation. Divide by the observation count `m`, not by the sum of discount weights; this is not an exponentially weighted average.

Project sampling convention: initialization `t0` installs references but is not a moment sample. Every subsequent one-minute raw reading contributes, including synchronization rows. Neither `S_n` nor `m` resets at synchronization, and `m` is not a synchronization count. Before the first sample, the implementation exposes zero moments to the separately labeled project allocation fallback.

Hand calculation for post-initialization readings A=`[2,4,6]`, B=`[-4,2,3]`:

| m | Discounted A observations | S_A | mu_A | Discounted B observations | S_B | mu_B |
|---|---|---|---|---|---|---|
| 1 | [2] | 2 | 2 | [-4] | -4 | -4 |
| 2 | [1,4] | 5 | 5/2 | [-2,2] | 0 | 0 |
| 3 | [0.5,2,6] | 8.5 | 8.5/3 | [-1,1,3] | 3 | 1 |

The separate clustering history is `h_n(t)=0.5*h_n(t-1)+(v_n(t)-reference_n)`, not a raw-reading moment and not divided by `m`. FMIC applies `abs(mu_n)` only when computing weights; signed first moments are preserved. The hand-calculated regression is `FirstMomentTests.test_discounted_arithmetic_mean_is_not_history_or_ema` in [the consolidated tests](phase3_test.py); synchronization persistence is checked separately by `test_history_and_first_moment_persist_across_sync`.

### SMIC second-moment provenance and stable recurrence

The **rendered** source section 1.4 (printed p.10/PDF 14) defines `sigma_n^2` as a sum of squared deviations without a divisor, but prints an inconsistent index `i=0..m`. [Appendix 1, printed p.22/PDF 26](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=26) resolves this: `sigma_n^2 = sum(i=1..m, (v'_n(t_i)-mu_n)^2)`. We use those `m` observations, with **no division by m or m-1**. Despite the source's term 'central second moment', this quantity is not conventional population/sample variance.

[Appendix 1, printed p.24/PDF 28](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=28) gives a stable update: with `k` previous observations, old mean `mu`, and new raw value `x`, `sigma_new^2 = 0.25*sigma_old^2 + k/(k+1)*(x-0.5*mu)^2`. It scales every previous vector by 1/2 and appends x. This reuses the unchanged first-moment update, keeps one common observation count, and never resets at root synchronization. Both terms are nonnegative, so no numerical cancellation correction or negative-value clamp is used; nonfinite or negative states fail explicitly.

Hand example A from above: at m=1, sigma^2=0 and W=2; at m=2, squared deviations `[9/4,9/4]` sum to `9/2`, giving `W=5/2+sqrt((1/2)*(9/2))=4`; at m=3, `[49/9,25/36,361/36]` sum to `97/6`, giving `W=17/6+sqrt(97/9)`. `SecondMomentTests.test_source_discounted_squared_deviations_and_weights_by_hand` verifies both sensors at every step.

SMIC uses `W_n=abs(mu_n)+sqrt(((m-1)/m)*sigma_n^2)`, then the same normalized node and mean coordinator allowances as FMIC. At m=0, zero moments and equal weights are an explicit project initialization convention; at m=1 the central sum and conservative contribution are zero. The approved project all-zero fallback applies only if the total weight is zero; an individual zero weight otherwise stays zero.

`FMICMonitor` retains its compatibility name but is the single lifecycle for `weight_method='FMIC'` or `'SMIC'`. Moment/weight payloads accompany existing sensor-to-root synchronization responses and allowances accompany existing bundled updates. No extra exchanges or internal-computation messages are charged. References, clustering, signed cancellation, escalation, next-bucket activation, and two-sided correctness policy are shared. Each method generates its own schedule and writes distinct compact outputs; FMIC schemas remain unchanged.

### Shared protocol and transmission units

The root is separate from sensor/coordinators. The coordinator is the
lexicographically greatest cluster member. Local absolute-deviation equality
triggers (`>=`); the coordinator takes absolute value **after** averaging signed
member deviations. The root allowance is the absolute distance between the
synchronized mean and P90, on either side of the threshold. At exact P90, zero
allowances cause checks even for unchanged readings on the following bucket.

Triggering non-coordinators report once; the coordinator requests/responses only
from peers not already reporting. Coordinator self-readings are free. Every
violating coordinator sends one root alert; multiple alerts cause one collection
per bucket. Alerts contain no raw readings, so root requests/retrieves all six
states and sends one bundled update per sensor. Requests, responses and updates
are distinct actual transmissions, not a sensor-as-root broadcast approximation.
Moments fit existing responses and allowances existing updates; internal
computation and scalar payload fields are not additional messages.

References, partition and allowances stay fixed until synchronization; the
synchronization reading updates history against entry references before
installing state for the next bucket. Global means use `math.fsum`. Centralized
truth is an offline oracle, never a hidden communication trigger. Continuous
US-3.24 snapshots allow only the established 1e-12 tolerance; triggers, states,
partitions, message counts and schedules remain exact.

## 3. Comparators and C0 equivalence

Keep the historical **signed-margin** Phase 2 result (38,087 messages; 2,107
synchronizations) separate from the revised **two-sided** Phase 2/C0 comparator
(9,159 messages; 467 synchronizations). The centralized 119,904 transmissions
are a different reference again. Historical percentages must not be relabeled
as savings against revised Phase 2.

C0 fixes equal allowances and singleton clusters, ignores alpha/moment values
for decisions, and is verified per bucket against revised Phase 2. C1 and C2
also fix singleton clusters but use FMIC and SMIC weights respectively. Controls
have no alpha trajectory. C0 may have been genuinely regenerated or reused only
with current provenance; the saved run explicitly records which occurred.

In [ ]:
comparator_table = pd.DataFrame([
    {"reference": "Historical signed-margin Phase 2 (not the primary denominator)",
     "actual_transmissions": historical["total_messages"],
     "root_synchronizations": historical["resynchronization_count"]},
    {"reference": "Revised two-sided Phase 2 / C0 (primary comparator)",
     "actual_transmissions": revised["total_messages"],
     "root_synchronizations": revised["resynchronization_count"]},
    {"reference": "Centralized transmission reference",
     "actual_transmissions": report["comparisons"]["centralized_reference"]["total_messages"],
     "root_synchronizations": pd.NA},
])
display(comparator_table)
display(pd.DataFrame([{
    "C0_reused": report["prerequisite_gates"]["C0_reused_from_US-3.24"],
    "per_bucket_equivalence": equivalence["comparison"]["passed"],
    "compared_buckets": equivalence["contract"]["bucket_count"],
    "messages": equivalence["message_mapping"]["phase3_total_messages"],
    "root_synchronizations": equivalence["message_mapping"]["root_synchronizations_including_initialization"],
    "event_sha256": equivalence["outputs"]["events_sha256"],
}]))

## 4. Weighting without clustering; matched clustering contrasts

C0 minus C1/C2 isolates weighting under the shared fixed-singleton policy.
C1 versus clustered FMIC and C2 versus clustered SMIC are the matched clustering
contrasts conditional on weighting. FMIC minus SMIC at equal alpha compares
complete methods, not second moments in isolation from their changed schedules.
The prespecified alpha is **0.10**. Tested-grid minima are exploratory, in-sample,
and must remain labeled separately.

In [ ]:
comparison_table = pd.DataFrame(report["configuration_comparison_table"])
comparison_columns = [
    "comparison_scope", "config_id", "alpha_label", "total_messages",
    "root_synchronization_count", "coordinator_check_count",
    "checks_resolved_without_escalation", "checks_escalated",
    "centralized_reduction_percent",
]
display(comparison_table[comparison_columns])
display(pd.json_normalize(report["comparisons"]["weighting_effects_without_clustering"],
                          sep=" / ").T.rename(columns={0: "saved value"}))
matched_rows = []
for method in ("FMIC", "SMIC"):
    for label in sorted({"0.10", *minima[method]["alpha_labels"]}):
        matched_rows.append({
            "algorithm": method, "alpha_label": label,
            "scope": "prespecified" if label == "0.10" else "exploratory in-sample minimum",
            **report["comparisons"]["clustering_effects_conditional_on_weighting"][method][label],
        })
display(pd.DataFrame(matched_rows))
display(pd.DataFrame(report["comparisons"]["full_fmic_minus_smic_at_same_alpha"]).T)

## 5. Existing interpretation and alpha sensitivity

The following calculation is relocated unchanged from the revised runner's
`_interpretation` function and uses **saved rows only**. It retains existing
ranges, direction reversals, matched contrasts, selected runs, pathways and
check proportions; it does not introduce new metrics or run an experiment.

In [ ]:
from typing import Any

def _interpretation(
    controls: list[dict[str, Any]],
    alphas: list[dict[str, Any]],
    revised: dict[str, Any],
    minima: dict[str, Any],
) -> dict[str, Any]:
    by_id = {row["config_id"]: row for row in controls}

    def selected_run(row: dict[str, Any]) -> dict[str, Any]:
        return {
            "alpha_label": row["alpha_label"],
            "total_messages": row["total_messages"],
            "root_synchronization_count": row["root_synchronization_count"],
            "revised_phase2_reduction_percent": row[
                "reduction_vs_revised_phase2_percent"
            ],
            "matched_singleton_message_difference": row[
                "matched_singleton_message_difference"
            ],
            "matched_singleton_reduction_percent": row[
                "matched_singleton_reduction_percent"
            ],
            "matched_singleton_root_synchronization_difference": row[
                "matched_singleton_root_synchronization_difference"
            ],
            "coordinator_check_count": row["coordinator_check_count"],
            "checks_resolved_without_escalation": row[
                "checks_resolved_without_escalation"
            ],
            "checks_escalated": row["checks_escalated"],
            "locally_resolved_check_share_percent": (
                100 * row["checks_resolved_without_escalation"]
                / row["coordinator_check_count"]
                if row["coordinator_check_count"] else None
            ),
            "repeated_coordinator_check_count": row[
                "repeated_coordinator_check_count"
            ],
            "repeated_check_share_percent": row["repeated_check_share_percent"],
            "repeated_checks_resolved_without_escalation": row[
                "repeated_checks_resolved_without_escalation"
            ],
            "repeated_locally_resolved_share_percent": (
                100 * row["repeated_checks_resolved_without_escalation"]
                / row["repeated_coordinator_check_count"]
                if row["repeated_coordinator_check_count"] else None
            ),
            "communication_pathways": {
                name: {
                    "messages": row[name],
                    "share_percent": row[f"{name}_share_percent"],
                }
                for name in (
                    "within_cluster_messages",
                    "coordinator_to_root_messages",
                    "root_wide_messages",
                )
            },
            "mean_cluster_count_per_synchronization": row[
                "cluster_count_mean_per_sync_event"
            ],
            "singleton_partition_event_share_percent": row[
                "singleton_partition_event_share_percent"
            ],
        }

    sensitivity: dict[str, Any] = {}
    for method in ("FMIC", "SMIC"):
        series = sorted(
            (row for row in alphas if row["algorithm"] == method),
            key=lambda row: float(row["alpha_label"]),
        )
        differences = [
            current["total_messages"] - previous["total_messages"]
            for previous, current in zip(series, series[1:])
        ]
        signs = [1 if value > 0 else -1 if value < 0 else 0 for value in differences]
        nonzero_signs = [sign for sign in signs if sign]
        reversals = sum(
            left != right
            for left, right in zip(nonzero_signs, nonzero_signs[1:])
        )
        alpha10 = next(row for row in series if row["alpha_label"] == "0.10")
        control = by_id["C1" if method == "FMIC" else "C2"]
        best_runs = [
            next(
                row for row in series
                if row["alpha_label"] == alpha_label
            )
            for alpha_label in minima[method]["alpha_labels"]
        ]
        sensitivity[method] = {
            "message_total_range": {
                "minimum": min(row["total_messages"] for row in series),
                "maximum": max(row["total_messages"] for row in series),
                "direction_reversals_across_tested_grid": reversals,
            },
            "root_synchronization_range": {
                "minimum": min(row["root_synchronization_count"] for row in series),
                "maximum": max(row["root_synchronization_count"] for row in series),
            },
            "weighting_only_contrast": {
                "C0_minus_weighted_singleton_messages": (
                    by_id["C0"]["total_messages"] - control["total_messages"]
                ),
                "percent_relative_to_C0": (
                    100
                    * (by_id["C0"]["total_messages"] - control["total_messages"])
                    / by_id["C0"]["total_messages"]
                    if by_id["C0"]["total_messages"] else None
                ),
            },
            "prespecified_alpha_0.10": selected_run(alpha10),
            "exploratory_minimum_runs": [
                selected_run(row) for row in best_runs
            ],
        }
    return {
        "message_sensitivity": sensitivity,
        "matched_revised_phase2_messages": revised["total_messages"],
        "communication_by_cluster_size_source": (
            "phase3_cluster_sizes.csv"
        ),
        "interpretive_cautions": [
            "Grid message totals are irregular when direction reversals are present; minima are exploratory and in-sample.",
            "A locally resolved coordinator check is not evidence that a root synchronization was counterfactually avoided.",
            "Persistent local-allowance triggers and new global threshold-state changes are distinct; global correctness is checked against the independent per-bucket oracle.",
            "Repeated checks describe current protocol behavior and are not evidence that any check can safely be skipped.",
            "Message totals do not measure bytes, latency, or bandwidth; no statistical significance or availability robustness is inferred.",
        ],
    }

interpretation = _interpretation(controls, alphas, revised, minima)
display(pd.json_normalize(interpretation["message_sensitivity"], sep=" / ").T.rename(columns={0: "saved-result analysis"}))

### Actual transmissions, root synchronizations, and cluster counts

These are the existing revised alpha charts. Fixed singleton controls are
references, not alpha-dependent runs. Fewer root synchronizations do not imply
fewer total transmissions once within-cluster checks are counted. The historical
signed-margin benchmark is **not** the dashed revised comparison below.

In [ ]:
definitions = (
    ("total_messages", "Actual transmissions", "phase3a_us325_messages_vs_alpha.png"),
    ("root_synchronization_count", "Root synchronizations (including initialization)", "phase3a_us325_synchronizations_vs_alpha.png"),
    ("cluster_count_mean_per_sync_event", "Mean clusters per synchronization event", "phase3a_us325_clustering_vs_alpha.png"),
)
for column, ylabel, name in definitions:
    fig, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
    chart_series[name] = {}
    for method, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
        data = alpha_frame.loc[alpha_frame["algorithm"] == method].sort_values("alpha")
        axis.plot(data["alpha"], data[column], marker="o", color=color, label=method)
        chart_series[name][method] = {"alpha": data["alpha"].tolist(), "values": data[column].tolist()}
    if column == "total_messages":
        axis.axhline(revised["total_messages"], color="#444444", linestyle="--", label="Revised Phase 2 (fixed benchmark)")
        for control in controls:
            if control["config_id"] in ("C1", "C2"):
                axis.axhline(control["total_messages"], color="#777777", linestyle=":", linewidth=1,
                             label=f'{control["config_id"]} singleton control')
    axis.set_xlabel("Alpha (full clustered runs only)")
    axis.set_ylabel(ylabel)
    axis.set_xticks([float(label) for label in ALPHA_LABELS])
    axis.grid(True, alpha=0.25)
    axis.legend()
    figures[name] = fig
    plt.show()

## 6. Communication composition, local checks, and repetition

The existing three pathways are within-cluster (trigger reports plus peer
requests/responses), coordinator-to-root alerts, and root-wide
requests/responses/updates. Their totals reconcile to actual transmissions.
The singleton bar chart remains alpha-independent.

In [ ]:
group_columns = ("within_cluster_messages", "coordinator_to_root_messages", "root_wide_messages")
colors = ("#4c78a8", "#f58518", "#54a24b")
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True, sharey=True)
composition_name = "phase3a_us325_communication_composition.png"
chart_series[composition_name] = {}
for axis, method in zip(axes, ("FMIC", "SMIC")):
    data = alpha_frame.loc[alpha_frame["algorithm"] == method].sort_values("alpha")
    axis.stackplot(data["alpha"], *(data[column] for column in group_columns),
                   labels=("Within-cluster", "Coordinator-to-root", "Root-wide"), colors=colors, alpha=0.85)
    chart_series[composition_name][method] = {column: data[column].tolist() for column in ("alpha", *group_columns)}
    axis.set_title(method)
    axis.set_xlabel("Alpha")
    axis.set_xticks([float(label) for label in ALPHA_LABELS])
    axis.grid(True, alpha=0.2)
axes[0].set_ylabel("Actual transmissions by pathway")
axes[1].legend(loc="upper right")
figures[composition_name] = fig
plt.show()

fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
singleton_name = "phase3a_us325_singleton_communication_composition.png"
labels = controls_frame["config_id"].tolist()
bottom = [0] * len(controls)
for column, label, color in zip(group_columns, ("Within-cluster", "Coordinator-to-root", "Root-wide"), colors):
    values = controls_frame[column].tolist()
    axis.bar(labels, values, bottom=bottom, label=label, color=color)
    bottom = [prior + value for prior, value in zip(bottom, values)]
axis.set_xlabel("Alpha-independent singleton control")
axis.set_ylabel("Actual transmissions")
axis.set_title("Communication composition by control")
axis.legend()
axis.grid(axis="y", alpha=0.2)
chart_series[singleton_name] = {column: controls_frame[column].tolist() for column in ("config_id", *group_columns)}
figures[singleton_name] = fig
plt.show()

### Existing check and cluster-size diagnostics

A locally resolved check is **not** a counterfactual root synchronization
avoided. Repeated checks mean adjacent buckets, unchanged membership, and no
intervening synchronization. They describe the current policy, not evidence that
checks can safely be skipped. Persistent allowance triggers and new global state
changes are distinct. Checks, internal computation, and messages are not
interchangeable units.

In [ ]:
check_columns = [
    "config_id", "alpha_label", "coordinator_check_count", "checks_resolved_without_escalation",
    "checks_escalated", "repeated_coordinator_check_count",
    "repeated_checks_resolved_without_escalation", "repeated_check_share_percent",
    "singleton_partition_event_share_percent",
]
display(comparison_table.loc[comparison_table["config_id"] != "revised_phase2", check_columns])
display(control_size_frame)
display(size_frame)
display(snapshots[["config_id", "alpha_label", "synchronization_id", "bucket_time_utc",
                   "selected_partition", "global_average", "p90_threshold",
                   "root_allowance", "moment_observation_count"]])

## 7. Preserved historical sweep charts (separate comparator)

The original US-3.22 exports used the signed-margin Phase 2 benchmark. They are
historical presentation, not current validation or evidence against revised
Phase 2. To retain their existing content without conflating denominators, the
following **opt-in** section reads the complete saved historical sweep; it never
launches it. Missing historical inputs fail if requested. No automatic fallback
from missing revised results is permitted.

In [ ]:
SHOW_HISTORICAL = os.environ.get("PHASE3_SHOW_HISTORICAL", "0") == "1"
if SHOW_HISTORICAL:
    historical_bundle = load_phase3_analysis_results(ROOT, OUTPUT_DIR, include_historical=True)
    history_rows = historical_bundle["historical_alphas"]
    input_hashes.update(historical_bundle["input_hashes"])
    historical_definitions = (
        ("total_messages", "Actual transmissions", "phase3a_messages_vs_alpha.png"),
        ("root_synchronization_count", "Root synchronizations (including initialization)", "phase3a_synchronizations_vs_alpha.png"),
        ("cluster_count_mean_per_sync_event", "Mean clusters per synchronization event", "phase3a_clustering_vs_alpha.png"),
    )
    for column, ylabel, name in historical_definitions:
        fig, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        chart_series[name] = {}
        for method, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
            data = history_rows.loc[history_rows["algorithm"] == method].sort_values("alpha")
            axis.plot(data["alpha"], data[column], marker="o", color=color, label=f"{method} (historical correctness passed)")
            chart_series[name][method] = {"alpha": data["alpha"].tolist(), "values": data[column].tolist()}
        if column == "total_messages":
            axis.axhline(historical["total_messages"], color="#444444", linestyle="--",
                         label="Historical signed-margin Phase 2 (fixed benchmark)")
        axis.set_xlabel("Alpha (tested grid: 0.05 to 0.95 by 0.05)")
        axis.set_ylabel(ylabel)
        axis.set_title(f"Historical Phase 3A fixed A/C/D/E/F/G cohort: {ylabel} vs alpha")
        axis.set_xticks([float(label) for label in ALPHA_LABELS])
        axis.grid(True, alpha=0.25)
        axis.legend()
        figures[name] = fig
        plt.show()
else:
    display(Markdown("Historical charts not requested. Set `PHASE3_SHOW_HISTORICAL=1` before kernel start to render the preserved three-chart view."))

## 8. Existing limitations, analytical gaps, and export policy

These are controlled within-dataset contrasts, not general causal estimates.
The grid minimum is exploratory and in-sample, not a universal optimum.
Messages do not measure bytes, latency, or bandwidth. No statistical significance,
out-of-sample benefit, general method ranking, or availability robustness is
inferred from this fixed six-sensor temperature cohort.

**Deferred analytical gaps (not filled here):** independent/out-of-sample tuning,
statistical uncertainty, byte/latency measurements, availability-varying cohorts,
and counterfactual protocol/check suppression require separate research stories.
This notebook adds no experiments, performance metrics, or claims.

The five revised PNG exports previously came from production; their content is
now inline here. The three historical PNGs retain their original identities and
separate benchmark when requested. Production no longer generates the revised
PNGs or detailed interpretive narrative. Historical files are preserved, not
deleted or silently rewritten. Optional exports must use a **new, separate**
directory and the same historical filenames. Default execution does not export.

In [ ]:
for caution in interpretation["interpretive_cautions"]:
    display(Markdown(f"- {caution}"))
for limitation in report["interpretation_limits"]:
    display(Markdown(f"- {limitation}"))

EXPORT_DIR = os.environ.get("PHASE3_ANALYSIS_EXPORT_DIR")
if EXPORT_DIR:
    export_path = Path(EXPORT_DIR)
    if not export_path.is_absolute():
        export_path = ROOT / export_path
    export_path = export_path.resolve()
    protected = [OUTPUT_DIR, ROOT / "artifacts" / "phase3_operational", ROOT / "artifacts" / "phase3_archive", ROOT / "scripts", ROOT / "config", ROOT / "Data", ROOT / ".git"]
    if (export_path == ROOT or ROOT.is_relative_to(export_path) or export_path == ROOT / "artifacts"
        or any(export_path == path or export_path.is_relative_to(path) for path in protected)
        or (export_path.is_relative_to(ROOT / "artifacts") and
            any(part.startswith(("phase3_baseline_", "phase3_extraction_", "phase3_production_", "phase3_testing_", "phase3_analysis_"))
                for part in export_path.relative_to(ROOT / "artifacts").parts))):
        raise ValueError("Analysis exports require a new directory outside saved outputs, source and migration evidence.")
    export_path.mkdir(parents=True, exist_ok=False)
    for name, figure in figures.items():
        figure.savefig(export_path / name, dpi=160)
    print(f"Explicit PNG exports: {export_path}")

for filename, expected in input_hashes.items():
    with Path(filename).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise RuntimeError(f"Analysis input changed during execution: {filename}")
print(f"Read-only analysis complete: {len(controls)} controls, {len(alphas)} clustered runs, {len(figures)} inline figures.")